# Notebook 04: Evaluation and Fusion

## Fusion strategy

**Inverse-variance** (point-wise):
$$\hat{y}_i = \frac{k_i/\sigma^2_{k,i} + m_i/\sigma^2_{m,i}}{1/\sigma^2_{k,i} + 1/\sigma^2_{m,i}}$$

**Kalman + Gaspari-Cohn localization** (full 6×6 covariance):
$$\hat{x} = x_k + B_{loc}(B_{loc}+R_{loc})^{-1}(x_m - x_k)$$
$$B_{loc,ij} = B_{ij} \cdot G(|i-j|/L), \quad L=2$$

Variances/covariances estimated on validation run residuals.
Gaspari-Cohn guarantees positive semidefiniteness (standard in EnKF).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config as cfg
from src.fusion import gaspari_cohn_matrix
from IPython.display import Image, display as disp

# Gaspari-Cohn matrix
GC = gaspari_cohn_matrix(n=6, L=cfg.GC_CORR_LENGTH)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(GC, ax=ax, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'Pt{i}' for i in range(1,7)],
            yticklabels=[f'Pt{i}' for i in range(1,7)])
ax.set_title(f'Gaspari-Cohn Localization Matrix (L={cfg.GC_CORR_LENGTH})')
plt.tight_layout(); plt.show()
print("Compact support: correlations = 0 for |i-j| >= 2*L = 4 index units")


## Full results table

In [ ]:
all_res = pd.read_csv(cfg.OUT_METRICS / 'all_results.csv')
pivot = all_res.pivot_table(index='label', columns='horizon', values='obs_rmse').round(5)
print("Observed-Point RMSE (primary metric):")
display(pivot.sort_values(1))


In [ ]:
# Best fusion comparison plot
disp(Image(str(cfg.OUT_FIGURES / '09_fusion_gru_h01.png')))


In [ ]:
# Fusion weights
disp(Image(str(cfg.OUT_FIGURES / '10_fusion_weights_transformer.png')))


## Key findings

1. **Fusion beats both components**: best fused model (transformer_fused_kgc, h=3)
   achieves obs_rmse = 0.00068, vs kinetic_prior 0.00108 (37% improvement)
   and transformer standalone 0.00427 (84% improvement).

2. **Point 6 (inlet)**: fusion weight for model ≈ 0 (all weight on kinetic).
   This makes sense: CO2 at the gas inlet is dominated by operating conditions
   that the kinetic model handles well, but the data-driven model sees less signal
   for given the small dataset.

3. **MAPE caution**: MAPE values are inflated at upper stages (Pt 2–5) where
   CO2 ≈ 0 (>95% absorbed at stage 1). Small absolute errors become large
   relative errors. RMSE is the more reliable metric here.

4. **Reconstruction advantage**: KAR reduces residual interpolation error 6–17×
   vs linear interpolation, providing cleaner training targets.

In [ ]:
# Final comparison bar chart
disp(Image(str(cfg.OUT_FIGURES / '12_final_comparison_h1.png')))


In [ ]:
# Per-point heatmap
disp(Image(str(cfg.OUT_FIGURES / '07_per_point_heatmap.png')))


## Limitations

- Full 6-point target profile is reconstructed (pseudo-label): observed-point RMSE
  is the only metric based on real AT400 measurements.
- SDAE distribution shift on val/test (only 6 training runs).
- Kinetic prior is precomputed (MATLAB/Simulink from original repo), not reproduced.
- MHE-based reconstruction (Chai et al. 2026) would be superior but requires
  online ODE solver (IPOPT/CasADi); beyond scope of this assignment.